<a href="https://colab.research.google.com/github/aruzhanusserbai/lab-05-AI-course/blob/main/lab05_harness.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 05 -- The harness, not just the model

Content: Agents, Memory and Skills, Harness and Reliable AI Systems · Narxoz University

You are a junior analyst at Dala Insight, a small fictional firm in Almaty that writes
economic briefs from regional statistics. Your colleague is a small open language model
that can call five functions on the firm's data. In this lab you **write its agent
loop**, measure how often it is right, add protective layers around it (a *harness*)
**one at a time**, and measure again.

**No API key. No cost.** One open model (`Qwen2.5-3B-Instruct`, about 6 GB, downloaded
once per Colab session) runs inside this notebook. Every region, indicator and number is
synthetic.

What is graded: this notebook, run from top to bottom with every ✏️ cell filled in
(9 of 15 points), and `CONCLUSION.md`, a short text **you type yourself, without any AI
tool** (6 of 15). Details at the end.

## Setup

1. **Runtime -> Change runtime type -> T4 GPU.** On a CPU the model is far too slow.
2. Run the next cell once. It installs the two Python packages Colab may not have in the
   right version.
3. Run the cells in order, one at a time. A ✏️ cell is a cell where **you** write inside
   the quotes. Write your prediction *before* you run the cell that reveals the answer:
   a prediction written after the result is worth nothing, to you or to the grader.

In [2]:
# Colab only: the two packages the lab was prepared with.
import subprocess
import sys

subprocess.run([sys.executable, "-m", "pip", "-q", "install",
                "transformers==5.18.0", "accelerate==1.15.0"], check=True)
print("installed")

installed


## Part 0 -- the data, the questions and the toolbox

The next cell fetches the two CSV files from the lab's GitHub repository. The cell after
it is the toolbox: the five tools, the model wrapper, the reference agent loop, the three
harness layers, the scoring. Read it; you will use every part of it, but you edit nothing
in it.

In [3]:
import os
import subprocess

REPO_URL = "https://github.com/unreal-kz/lab-05-AI-course.git"
if not os.path.exists("data/regional_stats.csv"):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)
    os.chdir("lab-05-AI-course")
print("working in", os.getcwd())

working in /content/lab-05-AI-course


In [4]:
"""Lab 05 toolbox: the Dala Insight data, five tools, a reference agent loop and the harness layers.

Pasted unchanged into one notebook cell. Everything the lab measures lives here, so the notebook
and the instructor's reference run execute the same code. All data is synthetic.

Layout: data and tools -> tool-call parsing -> answer scoring -> the questions -> the model ->
the agent loop with its harness layers -> the fixed workflow -> statistics.
"""

from __future__ import annotations

import ast
import csv
import json
import math
import operator
import re
import time
from collections import Counter
from collections.abc import Callable
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Any, Protocol

MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"
MAX_NEW_TOKENS = 256

SYSTEM_PROMPT = (
    "You are an analyst at Dala Insight, a small firm that writes economic briefs from regional "
    "statistics of Kazakhstan. All data is synthetic. Use the tools to look numbers up; never "
    "answer a number from memory. Report values in the unit the tool returns. When you are done, "
    "end your reply with one line: ANSWER: <value>. For a region question the value is the region "
    "name. Do all arithmetic with the calculate tool, not in your head. If the data does not exist, "
    "write ANSWER: NO DATA."
)


class ToolError(Exception):
    """A tool refused its arguments. The message is written for the model to read."""


# ----------------------------------------------------------------------------- data and tools


@dataclass(frozen=True)
class Dataset:
    """The two CSV files: yearly values per region and the indicator catalogue."""

    rows: tuple[dict[str, Any], ...]
    indicators: dict[str, dict[str, str]]

    @property
    def regions(self) -> list[str]:
        """Sorted region names."""
        return sorted({r["region"] for r in self.rows})


def load_dataset(data_dir: str | Path) -> Dataset:
    """Read ``regional_stats.csv`` and ``indicators.csv`` from ``data_dir``."""
    data_dir = Path(data_dir)
    with open(data_dir / "regional_stats.csv", newline="", encoding="utf-8") as f:
        rows = [dict(r, year=int(r["year"]), value=float(r["value"])) for r in csv.DictReader(f)]
    with open(data_dir / "indicators.csv", newline="", encoding="utf-8") as f:
        indicators = {r["indicator"]: r for r in csv.DictReader(f)}
    return Dataset(tuple(rows), indicators)


_OPS: dict[type, Callable[[float, float], float]] = {
    ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul, ast.Div: operator.truediv,
}


def _eval_arith(node: ast.AST) -> float:
    """Evaluate a parsed arithmetic expression: numbers, + - * / and unary minus only."""
    if isinstance(node, ast.Expression):
        return _eval_arith(node.body)
    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)) and not isinstance(node.value, bool):
        return float(node.value)
    if isinstance(node, ast.UnaryOp) and isinstance(node.op, ast.USub):
        return -_eval_arith(node.operand)
    if isinstance(node, ast.BinOp) and type(node.op) in _OPS:
        return _OPS[type(node.op)](_eval_arith(node.left), _eval_arith(node.right))
    raise ToolError("Only numbers, + - * / and parentheses are allowed in an expression.")


class Toolbox:
    """The five tools. Each returns a JSON string; bad arguments raise ``ToolError``."""

    def __init__(self, data: Dataset) -> None:
        self.data = data

    def run(self, name: str, arguments: Any) -> str:
        """Call tool ``name`` with ``arguments`` and return its JSON output.

        Raises:
            ToolError: the tool name or an argument is wrong (the message says how).
        """
        tool = getattr(self, f"tool_{name}", None) if re.fullmatch(r"[a-z_]+", name or "") else None
        if tool is None:
            raise ToolError(f"Unknown tool {name!r}. Tools: {', '.join(TOOL_NAMES)}.")
        if not isinstance(arguments, dict):
            raise ToolError("Tool arguments must be a JSON object.")
        try:
            result = tool(**arguments)
        except TypeError as exc:
            raise ToolError(f"Wrong arguments for {name}: {exc}") from exc
        return json.dumps(result, ensure_ascii=False, separators=(",", ":"))

    def _region(self, region: Any) -> str:
        for known in self.data.regions:
            if isinstance(region, str) and known.lower() == region.strip().lower():
                return known
        raise ToolError(f"Unknown region {region!r}. Call list_regions to see the valid names.")

    def _indicator(self, indicator: Any) -> str:
        if indicator not in self.data.indicators:
            raise ToolError(f"Unknown indicator {indicator!r}. Valid: {', '.join(sorted(self.data.indicators))}.")
        return indicator

    @staticmethod
    def _year(value: Any, label: str) -> int | None:
        if value is None:
            return None
        if isinstance(value, bool) or not isinstance(value, int):
            raise ToolError(f"{label} must be an integer year, got {value!r}.")
        return value

    def tool_list_regions(self) -> list[str]:
        """List every region and city of republican significance in the dataset."""
        return self.data.regions

    def tool_list_indicators(self) -> list[dict[str, str]]:
        """List the indicators with their units and definitions."""
        return list(self.data.indicators.values())

    def tool_get_series(self, region: Any, indicator: Any, start_year: Any = None, end_year: Any = None) -> list[dict]:
        """Yearly values of one indicator for one region; years with no data are absent."""
        region, indicator = self._region(region), self._indicator(indicator)
        lo, hi = self._year(start_year, "start_year"), self._year(end_year, "end_year")
        out = [
            {"year": r["year"], "value": r["value"], "unit": r["unit"]}
            for r in self.data.rows
            if r["region"] == region and r["indicator"] == indicator
            and (lo is None or r["year"] >= lo) and (hi is None or r["year"] <= hi)
        ]
        return sorted(out, key=lambda r: r["year"])

    def tool_compare_regions(self, indicator: Any, year: Any) -> list[dict]:
        """Rank all regions that have data for one indicator in one year, highest first."""
        indicator, year = self._indicator(indicator), self._year(year, "year")
        ranked = [
            {"region": r["region"], "value": r["value"], "unit": r["unit"]}
            for r in self.data.rows if r["indicator"] == indicator and r["year"] == year
        ]
        if not ranked:
            raise ToolError(f"No data for {indicator!r} in {year}.")
        return sorted(ranked, key=lambda r: r["value"], reverse=True)

    def tool_calculate(self, expression: Any) -> dict[str, float]:
        """Evaluate an arithmetic expression such as (120 - 100) / 100 * 100."""
        if not isinstance(expression, str):
            raise ToolError("expression must be a string.")
        try:
            value = _eval_arith(ast.parse(expression.strip(), mode="eval"))
        except SyntaxError as exc:
            raise ToolError(f"Cannot parse expression {expression!r}.") from exc
        except ZeroDivisionError as exc:
            raise ToolError("Division by zero.") from exc
        return {"result": round(value, 4)}


def _schema(name: str, description: str, properties: dict[str, dict], required: list[str]) -> dict:
    return {"type": "function", "function": {"name": name, "description": description,
            "parameters": {"type": "object", "properties": properties, "required": required}}}


TOOL_SCHEMAS = [
    _schema("list_regions", "List every region and city in the dataset.", {}, []),
    _schema("list_indicators", "List the indicators with their units and definitions.", {}, []),
    _schema("get_series", "Yearly values of one indicator for one region. Years with no data are absent.", {
        "region": {"type": "string", "description": "Region name exactly as list_regions prints it"},
        "indicator": {"type": "string", "description": "population, avg_monthly_wage, unemployment_rate, cpi_yoy or retail_trade"},
        "start_year": {"type": "integer"}, "end_year": {"type": "integer"}}, ["region", "indicator"]),
    _schema("compare_regions", "Rank ALL regions by one indicator in one year, highest first. Use it to find which region is highest or lowest.", {
        "indicator": {"type": "string"}, "year": {"type": "integer"}}, ["indicator", "year"]),
    _schema("calculate", "Evaluate arithmetic: numbers, + - * / and parentheses.", {
        "expression": {"type": "string", "description": "for example (120 - 100) / 100 * 100"}}, ["expression"]),
]
TOOL_NAMES = [s["function"]["name"] for s in TOOL_SCHEMAS]


# ----------------------------------------------------------------------------- tool-call parsing

_CALL_RE = re.compile(r"<tool_call>\s*(.*?)\s*(?:</tool_call>|$)", re.DOTALL)


@dataclass(frozen=True)
class ToolCall:
    """One tool call found in a model reply. ``error`` is set when the JSON was not usable."""

    name: str
    arguments: Any
    error: str | None = None


def parse_tool_calls(text: str) -> list[ToolCall]:
    """Find the ``<tool_call>`` blocks in a reply (Qwen's format: a JSON object with name and arguments).

    A block whose JSON cannot be read comes back as a ``ToolCall`` with ``error`` set, so the caller
    can tell "the model wrote no call" from "the model wrote a broken call".
    """
    calls = []
    for body in _CALL_RE.findall(text):
        try:
            obj = json.loads(body)
        except json.JSONDecodeError as exc:
            calls.append(ToolCall("", None, f"Tool call is not valid JSON ({exc.msg})."))
            continue
        if not isinstance(obj, dict) or "name" not in obj:
            calls.append(ToolCall("", None, 'Tool call must be a JSON object with "name" and "arguments".'))
            continue
        calls.append(ToolCall(str(obj["name"]), obj.get("arguments", {})))
    return calls


# ----------------------------------------------------------------------------- numbers and answers

_NUM_RE = re.compile(r"(?<![\w.])-?(?:\d{1,3}(?:,\d{3})+|\d+)(?:\.\d+)?")


def numbers_in(text: str) -> list[str]:
    """Numbers as written in ``text`` ("1,234.5" counts as one); thousands commas kept."""
    return _NUM_RE.findall(text)


def to_float(token: str) -> float:
    """Parse a number token from ``numbers_in``."""
    return float(token.replace(",", ""))


def extract_answer(text: str) -> str | None:
    """The value after the last ``ANSWER:`` line, or ``None`` if there is no such line."""
    found = re.findall(r"^\s*ANSWER:\s*(.+?)\s*$", text, flags=re.MULTILINE | re.IGNORECASE)
    return found[-1] if found else None


def is_no_data(answer: str) -> bool:
    """True for the agreed "the data does not exist" answer."""
    return re.sub(r"[^a-z ]", "", answer.lower()).strip() == "no data"


def is_grounded(answer: str, tool_outputs: list[str]) -> tuple[bool, str]:
    """Does every value in ``answer`` appear in some tool output? Returns ``(ok, reason)``.

    Numbers are compared after rounding the tool value to as many decimals as the answer shows, so
    "95.2" matches 95.1999. Years (1900-2100 written as integers) are exempt. A name must occur in
    a tool output as text. "NO DATA" is grounded when a tool returned an empty list or "No data".
    """
    joined = " ".join(tool_outputs)
    if is_no_data(answer):
        empty = any(o.strip() in ("[]", "") or o.startswith("Error: No data") for o in tool_outputs)
        return (empty, "" if empty else "no tool returned an empty result, so NO DATA is not supported")
    tool_numbers = [to_float(t) for o in tool_outputs for t in numbers_in(o)]
    tokens = numbers_in(answer)
    if not tokens:
        ok = answer.strip().lower() in joined.lower()
        return (ok, "" if ok else f"{answer!r} does not occur in any tool output")
    for token in tokens:
        value = to_float(token)
        if "." not in token and 1900 <= value <= 2100:
            continue
        decimals = len(token.split(".")[1]) if "." in token else 0
        if not any(round(t, decimals) == round(value, decimals) for t in tool_numbers):
            return (False, f"the number {token} is not in any tool output")
    return (True, "")


@dataclass(frozen=True)
class Question:
    """A question with the answer computed from the data (never typed by hand)."""

    qid: str
    kind: str            # lookup | rank | growth | pp | nodata
    text: str
    expected: Any        # float, region name, or None for "no data"
    tol: float = 0.0
    shape: str = "open"  # "fixed" if a one-call workflow can answer it
    params: dict[str, Any] = field(default_factory=dict)


def score_answer(q: Question, answer: str | None) -> str:
    """``correct``, ``wrong`` or ``none`` (no usable answer). A refusal to answer is ``none``, not wrong.

    Scored by content, so a reply in a sentence counts as long as it contains the expected value:
    a number matches if it is within ``q.tol`` (years written as 1900-2100 are ignored); a region
    must appear by its full name; "no data" is correct only for the questions that have none.
    """
    if answer is None or not answer.strip() or answer.strip().upper() == "UNVERIFIED":
        return "none"
    if q.expected is None:
        return "correct" if re.search(r"\bno data\b", answer, flags=re.IGNORECASE) else "wrong"
    if re.search(r"\bno data\b", answer, flags=re.IGNORECASE):
        return "wrong"
    if isinstance(q.expected, str):
        return "correct" if re.search(rf"\b{re.escape(q.expected)}\b", answer, flags=re.IGNORECASE) else "wrong"
    values = [to_float(t) for t in numbers_in(answer) if not ("." not in t and 1900 <= to_float(t) <= 2100)]
    return "correct" if any(abs(v - q.expected) <= q.tol for v in values) else "wrong"


_LABEL = {
    "population": "population", "avg_monthly_wage": "average monthly wage",
    "unemployment_rate": "unemployment rate", "cpi_yoy": "consumer price inflation (CPI, year on year)",
    "retail_trade": "retail trade turnover",
}


def build_questions(data: Dataset) -> list[Question]:
    """The 24 questions of the lab; every expected value is computed from ``data``."""
    tb = Toolbox(data)

    def value(region: str, ind: str, year: int) -> float:
        series = tb.tool_get_series(region, ind, year, year)
        return series[0]["value"]

    def pick(ind: str, year: int, *, highest: bool) -> str:
        ranked = tb.tool_compare_regions(ind, year)
        top = ranked[0] if highest else ranked[-1]
        assert sum(r["value"] == top["value"] for r in ranked) == 1, "tie in a rank question"
        return top["region"]

    qs: list[Question] = []
    for qid, region, ind, year in [
        ("q01", "Karaganda", "avg_monthly_wage", 2024), ("q02", "Shymkent", "population", 2025),
        ("q03", "Atyrau", "unemployment_rate", 2023), ("q04", "Kostanay", "cpi_yoy", 2022),
        ("q05", "Astana", "retail_trade", 2025), ("q06", "Almaty City", "population", 2025),
    ]:
        qs.append(Question(qid, "lookup", f"What was the {_LABEL[ind]} in {region} in {year}?",
                           value(region, ind, year), 0.06, "fixed",
                           {"region": region, "indicator": ind, "year": year}))
    for qid, ind, year, hi in [
        ("q07", "unemployment_rate", 2025, True), ("q08", "avg_monthly_wage", 2024, False),
        ("q09", "population", 2023, True), ("q10", "cpi_yoy", 2022, True), ("q11", "retail_trade", 2021, False),
    ]:
        word = "highest" if hi else "lowest"
        qs.append(Question(qid, "rank", f"Which region had the {word} {_LABEL[ind]} in {year}? Answer with the region name.",
                           pick(ind, year, highest=hi), 0.0, "fixed",
                           {"indicator": ind, "year": year, "order": word}))
    for qid, region, ind, y0, y1 in [
        ("q12", "Pavlodar", "avg_monthly_wage", 2020, 2025), ("q13", "Mangystau", "retail_trade", 2021, 2025),
        ("q14", "Turkistan", "population", 2019, 2025), ("q15", "Kyzylorda", "avg_monthly_wage", 2022, 2024),
    ]:
        a, b = value(region, ind, y0), value(region, ind, y1)
        qs.append(Question(qid, "growth",
                           f"By how many percent did the {_LABEL[ind]} in {region} change between {y0} and {y1}? "
                           "Give the percent change with one decimal.",
                           round((b - a) / a * 100, 1), 0.15, "open"))
    for qid, region, ind, y0, y1 in [
        ("q16", "Astana", "unemployment_rate", 2021, 2025), ("q17", "Aktobe", "unemployment_rate", 2020, 2023),
        ("q18", "Jambyl", "cpi_yoy", 2022, 2025),
    ]:
        diff = round(value(region, ind, y1) - value(region, ind, y0), 1)
        assert diff != 0
        qs.append(Question(qid, "pp",
                           f"By how many percentage points did the {_LABEL[ind]} in {region} "
                           f"{'rise' if diff > 0 else 'fall'} between {y0} and {y1}?",
                           abs(diff), 0.06, "open"))
    for qid, region, ind, year in [("q19", "Ulytau", "avg_monthly_wage", 2020), ("q20", "Abai", "unemployment_rate", 2019)]:
        assert not tb.tool_get_series(region, ind, year, year)
        qs.append(Question(qid, "nodata", f"What was the {_LABEL[ind]} in {region} in {year}?", None, 0.0, "open"))
    # Names as people really write them: an old name, a transliteration, a descriptive phrase. The tools
    # accept only the dataset's names, so the first call fails and the agent has to look the name up.
    for qid, alias, canonical, ind, year in [
        ("q21", "Nur-Sultan", "Astana", "unemployment_rate", 2022),
        ("q22", "Chimkent", "Shymkent", "avg_monthly_wage", 2023),
        ("q23", "Karagandy", "Karaganda", "retail_trade", 2024),
        ("q24", "the city of Almaty", "Almaty City", "population", 2024),
    ]:
        assert alias not in data.regions and canonical in data.regions
        qs.append(Question(qid, "alias", f"What was the {_LABEL[ind]} in {alias} in {year}?",
                           value(canonical, ind, year), 0.06, "fixed",
                           {"region": canonical, "indicator": ind, "year": year, "alias": alias}))
    return qs


# ----------------------------------------------------------------------------- the model


@dataclass(frozen=True)
class Generation:
    """One model reply and what it cost."""

    text: str
    prompt_tokens: int
    new_tokens: int


class Model(Protocol):
    """Anything that turns chat messages into a reply (the real model, or a script in tests)."""

    def generate(self, messages: list[dict], *, use_tools: bool = True,
                 max_new_tokens: int = MAX_NEW_TOKENS) -> Generation: ...


def pick_device() -> str:
    """``cuda`` if present, else Apple ``mps``, else ``cpu``."""
    import torch

    if torch.cuda.is_available():
        return "cuda"
    if torch.backends.mps.is_available():
        return "mps"
    return "cpu"


class LocalLLM:
    """``Qwen2.5-Instruct`` in eval mode, greedy decoding: the same messages give the same reply."""

    def __init__(self, model_id: str = MODEL_ID, device: str | None = None) -> None:
        import torch
        from transformers import AutoModelForCausalLM, AutoTokenizer

        self.device = device or pick_device()
        dtype = torch.float32 if self.device == "cpu" else torch.float16
        self.tokenizer = AutoTokenizer.from_pretrained(model_id)
        self.model = AutoModelForCausalLM.from_pretrained(model_id, dtype=dtype).to(self.device)
        self.model.eval()

    def generate(self, messages: list[dict], *, use_tools: bool = True,
                 max_new_tokens: int = MAX_NEW_TOKENS) -> Generation:
        """Reply to ``messages``; the tool schemas are put into the prompt when ``use_tools``."""
        import torch

        prompt = self.tokenizer.apply_chat_template(
            messages, tools=TOOL_SCHEMAS if use_tools else None, add_generation_prompt=True, tokenize=False)
        inputs = self.tokenizer(prompt, return_tensors="pt").to(self.device)
        with torch.inference_mode():
            # repetition_penalty=1.0: the checkpoint's default is 1.1, which punishes copying a digit twice.
            out = self.model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                                      repetition_penalty=1.0, temperature=None, top_p=None, top_k=None)
        new_ids = out[0][inputs["input_ids"].shape[1]:]
        text = self.tokenizer.decode(new_ids, skip_special_tokens=True)
        return Generation(text, int(inputs["input_ids"].shape[1]), int(new_ids.shape[0]))


# ----------------------------------------------------------------------------- the agent loop


def build_messages(question: str, notes: str = "") -> list[dict]:
    """The starting conversation: system prompt (plus the firm's notes, if any) and the question."""
    system = SYSTEM_PROMPT + (f"\n\nFirm notes:\n{notes.strip()}" if notes.strip() else "")
    return [{"role": "system", "content": system}, {"role": "user", "content": question}]


@dataclass(frozen=True)
class Harness:
    """Which protective layers are on. ``BARE`` is the loop with none of them."""

    validate: bool = False   # layer 1: a bad call goes back to the model as an error message
    limit: bool = False      # layer 2: tight step budget, repeated calls cut, a forced final answer
    ground: bool = False     # layer 3: the answer line must exist and every value must come from a tool
    max_steps: int = 8
    max_fixes: int = 2       # how many times a layer may send the model back before giving up


BARE = Harness()
LAYER_1 = Harness(validate=True)
LAYER_2 = Harness(validate=True, limit=True, max_steps=5)
LAYER_3 = Harness(validate=True, limit=True, ground=True, max_steps=5)


@dataclass
class RunResult:
    """Everything one question left behind: the answer, the verdict, the cost and the trace."""

    qid: str
    answer: str | None
    outcome: str                      # correct | wrong | none
    failure: str                      # "" or a tag from FAILURES
    model_calls: int
    prompt_tokens: int
    new_tokens: int
    seconds: float
    trace: list[dict]

    def to_dict(self) -> dict[str, Any]:
        """JSON-ready form."""
        return asdict(self)


FAILURES = ("bad_json", "unknown_tool", "bad_args", "step_limit", "no_answer_line", "ungrounded")


def run_agent(model: Model, toolbox: Toolbox, q: Question, harness: Harness = BARE, notes: str = "") -> RunResult:
    """The reference loop. The student's ``my_agent`` in Part 2 is the ``BARE`` path of this function.

    Loop: ask the model; no tool call means that reply is the final answer; otherwise run every call,
    append the results and ask again. The layers in ``harness`` change what happens when something
    goes wrong, never the happy path. ``failure`` is the first thing that went wrong in the run,
    even if a layer then recovered from it.
    """
    t0 = time.perf_counter()
    messages = build_messages(q.text, notes)
    trace: list[dict] = []
    evidence: list[str] = []          # tool outputs the answer may rely on
    seen: set[str] = set()
    count = {"calls": 0, "prompt": 0, "new": 0, "fixes": 0}
    failure = ""

    def ask(use_tools: bool = True) -> str:
        gen = model.generate(messages, use_tools=use_tools)
        count["calls"] += 1
        count["prompt"] += gen.prompt_tokens
        count["new"] += gen.new_tokens
        messages.append({"role": "assistant", "content": gen.text})
        trace.append({"kind": "model", "text": gen.text})
        return gen.text

    def send(kind: str, role: str, text: str) -> None:
        trace.append({"kind": kind, "text": text})
        messages.append({"role": role, "content": text})

    def done(final: str | None) -> RunResult:
        return RunResult(q.qid, final, score_answer(q, final), failure, count["calls"], count["prompt"],
                         count["new"], round(time.perf_counter() - t0, 2), trace)

    for step in range(1, harness.max_steps + 1):
        text = ask()
        calls = parse_tool_calls(text)

        if not calls:                                           # the model replied without a tool call
            if "<tool_call>" in text:
                failure = failure or "bad_json"
            candidate = extract_answer(text)
            if candidate is None:
                failure = failure or "no_answer_line"
            if harness.ground:
                problem = ""
                if candidate is None:
                    problem = "Your reply has no line 'ANSWER: <value>'. Finish with that line."
                else:
                    ok, why = is_grounded(candidate, evidence)
                    if not ok:
                        failure = failure or "ungrounded"
                        problem = (f"Your answer is not supported by the tool results: {why}. "
                                   "Use tool results; if the value comes from arithmetic, compute it with the calculate tool. "
                                   "If the data does not exist, answer ANSWER: NO DATA.")
                if problem:
                    if count["fixes"] >= harness.max_fixes:
                        return done("UNVERIFIED")
                    count["fixes"] += 1
                    send("reject", "user", problem)
                    continue
            return done(candidate if candidate is not None else text.strip())

        repeated = False
        for call in calls:                                      # run every call the model wrote
            ok = False
            if call.error:
                result, tag = f"Error: {call.error}", "bad_json"
            else:
                try:
                    result, tag, ok = toolbox.run(call.name, call.arguments), "", True
                    evidence.append(result)
                except ToolError as exc:
                    result = f"Error: {exc}"
                    tag = "unknown_tool" if str(exc).startswith("Unknown tool") else "bad_args"
                    if str(exc).startswith("No data"):
                        evidence.append(result)
            failure = failure or tag
            trace.append({"kind": "tool", "name": call.name, "arguments": call.arguments, "ok": ok, "text": result})
            if not ok and not harness.validate:
                return done(None)                               # layer 0: the program falls over here
            messages.append({"role": "tool", "content": result})
            key = json.dumps([call.name, call.arguments], sort_keys=True, default=str)
            repeated = repeated or key in seen
            seen.add(key)

        if harness.limit and (repeated or step == harness.max_steps):
            failure = failure or "step_limit"
            send("reject", "user", "Stop calling tools. Write the final reply now, using only the tool "
                                   "results above, ending with ANSWER: <value>.")
            candidate = extract_answer(ask(use_tools=False))
            if candidate is not None and harness.ground and not is_grounded(candidate, evidence)[0]:
                failure = failure or "ungrounded"
                candidate = "UNVERIFIED"
            return done(candidate)

    failure = failure or "step_limit"                           # budget spent, no final reply
    return done(None)


def _finish(q: Question, answer: str | None, failure: str, calls: int, p_tok: int, n_tok: int,
            t0: float, trace: list[dict]) -> RunResult:
    return RunResult(q.qid, answer, score_answer(q, answer), failure, calls, p_tok, n_tok,
                     round(time.perf_counter() - t0, 2), trace)


# ----------------------------------------------------------------------------- the fixed workflow

WORKFLOW_PROMPT = (
    "Read the question and reply with ONE JSON object and nothing else.\n"
    'For a lookup of one value: {"kind":"lookup","region":"<region>","indicator":"<indicator>","year":<year>}\n'
    'For the region with the highest or lowest value: {"kind":"rank","indicator":"<indicator>","year":<year>,"order":"highest" or "lowest"}\n'
    "indicator is one of: population, avg_monthly_wage, unemployment_rate, cpi_yoy, retail_trade.\n"
    "Region names: Abai, Akmola, Aktobe, Almaty City, Almaty Region, Astana, Atyrau, East Kazakhstan, Jambyl, "
    "Jetisu, Karaganda, Kostanay, Kyzylorda, Mangystau, North Kazakhstan, Pavlodar, Shymkent, Turkistan, "
    "Ulytau, West Kazakhstan.\n"
    "Question: "
)


def run_workflow(model: Model, toolbox: Toolbox, q: Question) -> RunResult:
    """Fixed pipeline: one model call extracts the parameters, plain code does the rest.

    No loop, no choice of tools. If the reply is not a usable JSON object the result is ``none``.
    """
    t0 = time.perf_counter()
    gen = model.generate([{"role": "user", "content": WORKFLOW_PROMPT + q.text}], use_tools=False, max_new_tokens=80)
    trace = [{"step": 1, "kind": "model", "text": gen.text}]
    answer, failure = None, ""
    match = re.search(r"\{.*\}", gen.text, flags=re.DOTALL)
    try:
        spec = json.loads(match.group(0)) if match else None
        if not isinstance(spec, dict):
            raise TypeError("no JSON object")
        if spec.get("kind") == "lookup":
            rows = json.loads(toolbox.run("get_series", {"region": spec["region"], "indicator": spec["indicator"],
                                                         "start_year": spec["year"], "end_year": spec["year"]}))
            answer = str(rows[0]["value"]) if rows else "NO DATA"
        elif spec.get("kind") == "rank":
            ranked = json.loads(toolbox.run("compare_regions", {"indicator": spec["indicator"], "year": spec["year"]}))
            answer = (ranked[0] if spec.get("order") == "highest" else ranked[-1])["region"]
        else:
            raise ValueError("unsupported kind")
    except (ValueError, TypeError, KeyError, ToolError) as exc:
        failure = "bad_args" if isinstance(exc, ToolError) else "bad_json"
        trace.append({"step": 1, "kind": "tool", "ok": False, "text": f"Error: {exc}"})
    return _finish(q, answer, failure, 1, gen.prompt_tokens, gen.new_tokens, t0, trace)


# ----------------------------------------------------------------------------- statistics


def wilson(successes: int, n: int, z: float = 1.96) -> tuple[float, float]:
    """Wilson score interval for a proportion (95 % for ``z=1.96``); stays inside [0, 1]."""
    if n == 0:
        return (0.0, 1.0)
    p = successes / n
    denom = 1 + z * z / n
    centre = (p + z * z / (2 * n)) / denom
    half = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / denom
    return (max(0.0, centre - half), min(1.0, centre + half))


def summarize(results: list[RunResult]) -> dict[str, Any]:
    """Counts, Wilson interval, cost and failure tags for one configuration."""
    n = len(results)
    correct = sum(r.outcome == "correct" for r in results)
    lo, hi = wilson(correct, n)
    return {
        "n": n, "correct": correct, "wrong": sum(r.outcome == "wrong" for r in results),
        "none": sum(r.outcome == "none" for r in results), "ci95": (round(lo, 3), round(hi, 3)),
        "model_calls": sum(r.model_calls for r in results),
        "tokens": sum(r.prompt_tokens + r.new_tokens for r in results),
        "seconds": round(sum(r.seconds for r in results), 1),
        "failures": dict(Counter(r.failure for r in results if r.failure)),
    }


# ----------------------------------------------------------------------------- reports and checks

_SYMBOL = {"correct": "ok", "wrong": "WRONG", "none": "-"}


def summary_line(label: str, results: list[RunResult]) -> str:
    """One printed line per configuration; the notebook and the reference run use the same format."""
    s = summarize(results)
    return (f"{label:14s} correct {s['correct']:2d}/{s['n']}  wrong {s['wrong']:2d}  no answer {s['none']:2d}  "
            f"95% [{s['ci95'][0]:.2f}, {s['ci95'][1]:.2f}]  model calls {s['model_calls']:3d}  "
            f"tokens {s['tokens']:6d}  seconds {s['seconds']:6.1f}")


def outcome_table(runs: dict[str, list[RunResult]], questions: list[Question]) -> str:
    """Outcome of every question under every configuration (``ok``, ``WRONG`` or ``-`` for no answer)."""
    lines = [f"{'id':4s} {'kind':7s} " + " ".join(f"{name:>9s}" for name in runs)]
    for i, q in enumerate(questions):
        lines.append(f"{q.qid:4s} {q.kind:7s} " + " ".join(f"{_SYMBOL[res[i].outcome]:>9s}" for res in runs.values()))
    return "\n".join(lines)


def _flat(text: str, limit: int) -> str:
    return text.replace("\n", " ")[:limit]


def format_trace(result: RunResult) -> str:
    """The run as readable text: what the model said, which tools ran, what the harness answered."""
    head = f"{result.qid}: answer {result.answer!r} -> {result.outcome}"
    lines = [head + (f" (first thing that went wrong: {result.failure})" if result.failure else "")]
    for e in result.trace:
        if e["kind"] == "tool":
            args = json.dumps(e.get("arguments"), ensure_ascii=False)
            lines.append(f"  tool   {e.get('name', 'workflow')}({args}) -> "
                         f"{'ok' if e['ok'] else 'ERROR'}: {_flat(e['text'], 150)}")
        elif e["kind"] == "reject":
            lines.append(f"  harness: {_flat(e['text'], 200)}")
        else:
            lines.append(f"  model: {_flat(e['text'], 200)}")
    return "\n".join(lines)


def check_loop(agent: Callable[[str], tuple[str, list[dict]]], model: Model, toolbox: Toolbox,
               questions: list[Question], qids: tuple[str, ...] = ("q01", "q07", "q12")) -> bool:
    """Compare the student's ``agent(question) -> (final reply, messages)`` with the reference ``BARE`` loop.

    Decoding is greedy, so a correct loop produces the same final reply and the same number of tool
    results as the reference on every question. Returns True when all questions match.
    """
    by_id = {q.qid: q for q in questions}
    all_ok = True
    for qid in qids:
        ref = run_agent(model, toolbox, by_id[qid], BARE)
        ref_reply = next(e["text"] for e in reversed(ref.trace) if e["kind"] == "model")
        ref_tools = sum(e["kind"] == "tool" for e in ref.trace)
        try:
            reply, messages = agent(by_id[qid].text)
        except Exception as exc:  # noqa: BLE001 - the student's code may fail in any way; report it, do not hide it
            print(f"{qid}: your loop raised {type(exc).__name__}: {exc}")
            all_ok = False
            continue
        tools = sum(m["role"] == "tool" for m in messages)
        same = reply.strip() == ref_reply.strip() and tools == ref_tools
        all_ok = all_ok and same
        print(f"{qid}: {'MATCHES' if same else 'DIFFERS FROM'} the reference loop "
              f"(tool results: yours {tools}, reference {ref_tools})")
        if not same:
            print(f"   yours:     {reply.strip()[:150]!r}\n   reference: {ref_reply.strip()[:150]!r}")
    print("all three match" if all_ok else "not yet: read the lines above, then fix the loop")
    return all_ok


def notes_leak(notes: str, questions: list[Question]) -> list[str]:
    """Question ids whose numeric answer appears in ``notes``. Notes may hold conventions, never answers."""
    written = [to_float(t) for t in numbers_in(notes)]
    return [q.qid for q in questions
            if isinstance(q.expected, float) and any(abs(w - q.expected) <= q.tol for w in written)]

24 questions, each with an answer **computed from the CSV** (never typed by hand):

- `lookup`: the value of one indicator in one region and year;
- `rank`: which region is highest or lowest;
- `growth` and `pp`: a percent change, and a change in percentage points;
- `nodata`: a question whose data does not exist (three regions were created in 2022);
- `alias`: a region named the way people really write it (an old name, another
  spelling). **The tools accept only the names in `list_regions`.**

You see the questions below, not the answers.

In [5]:
from collections import Counter

DATA = load_dataset("data")
QUESTIONS = build_questions(DATA)
print(len(DATA.rows), "rows,", len(DATA.regions), "regions, indicators:", sorted(DATA.indicators))
years = {r: sorted({row["year"] for row in DATA.rows if row["region"] == r}) for r in DATA.regions}
print("regions whose series do not start in 2019:",
      {r: (y[0], y[-1]) for r, y in years.items() if y[0] != 2019})
print()
for q in QUESTIONS:
    print(f"{q.qid} {q.kind:7s} {q.text}")
print("\nkinds:", dict(Counter(q.kind for q in QUESTIONS)))

655 rows, 20 regions, indicators: ['avg_monthly_wage', 'cpi_yoy', 'population', 'retail_trade', 'unemployment_rate']
regions whose series do not start in 2019: {'Abai': (2022, 2025), 'Jetisu': (2022, 2025), 'Ulytau': (2022, 2025)}

q01 lookup  What was the average monthly wage in Karaganda in 2024?
q02 lookup  What was the population in Shymkent in 2025?
q03 lookup  What was the unemployment rate in Atyrau in 2023?
q04 lookup  What was the consumer price inflation (CPI, year on year) in Kostanay in 2022?
q05 lookup  What was the retail trade turnover in Astana in 2025?
q06 lookup  What was the population in Almaty City in 2025?
q07 rank    Which region had the highest unemployment rate in 2025? Answer with the region name.
q08 rank    Which region had the lowest average monthly wage in 2024? Answer with the region name.
q09 rank    Which region had the highest population in 2023? Answer with the region name.
q10 rank    Which region had the highest consumer price inflation (CPI, year o

✏️ **Predict 0, before you load the model.** In the next cell, write your guess in your
own words: (1) what share of these 24 questions will the plain agent of Part 1 answer
correctly; (2) which kinds of question will it fail, and why. Nothing grades whether you
are right; you will compare it with the measurement.

In [6]:
PREDICTION_0 = """(1)I predict that plain agent will answer correctly about 19-20 questions out of 24.
(2)I think it will fail on alias questions because the questions contain wrong names for the regions,
for example, in the q22 it says Chimkent but in the data it is written as Shymkent.
Same for the questions that says Nur-Sultan, instead of Astana, Karagandy not Karaganda, and etc.
Plain agent will stop even for the smallest mistakes, so that's why I think miswritten names of regions may cause the fail.
Also, regions Ulytau, Abai and Jetisu has data only from 2022. Questions 19 and 20 are asking for the data from 2020
for Ulytau and 2019 for Abai, but model doesn't have these. If it does not have specific data that question is asking for the tool will
return empty list, so the model will answer no data instead of making up a number. That counts as correct, so I didn’t count them as fails."""  # ✏️ write inside the quotes, then run this cell

## Part 1 -- a workflow and an agent, side by side

The next cell loads the model (the first time in a session this downloads about 6 GB),
puts it on the GPU, and prints the system prompt it will be given and the names of the
tools it may call. Decoding is greedy, so the same messages always give the same reply.

In [7]:
llm = LocalLLM()
toolbox = Toolbox(DATA)
print("model:", MODEL_ID, "| device:", llm.device, "| dtype:", llm.model.dtype,
      "| training mode:", llm.model.training)
if llm.device == "cpu":
    print("WARNING: no GPU found. Switch the runtime to T4 GPU (Setup, step 1) or this will take hours.")
print("\nsystem prompt:\n" + SYSTEM_PROMPT)
print("\ntools:", TOOL_NAMES)

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

model: Qwen/Qwen2.5-3B-Instruct | device: cuda | dtype: torch.float16 | training mode: False

system prompt:
You are an analyst at Dala Insight, a small firm that writes economic briefs from regional statistics of Kazakhstan. All data is synthetic. Use the tools to look numbers up; never answer a number from memory. Report values in the unit the tool returns. When you are done, end your reply with one line: ANSWER: <value>. For a region question the value is the region name. Do all arithmetic with the calculate tool, not in your head. If the data does not exist, write ANSWER: NO DATA.

tools: ['list_regions', 'list_indicators', 'get_series', 'compare_regions', 'calculate']


Two ways to answer a question with the same model and the same tools:

- A **workflow** is a fixed pipeline. One model call turns the question into a small JSON
  object (which indicator, which year, ...); ordinary code then calls the right tool and
  writes the answer from a template. No loop, no choice of tools.
- An **agent** is a loop. The model chooses a tool, sees the result, chooses again, and
  stops when it decides to answer. `run_agent(..., BARE)` is the plain version: no
  protection at all.

A workflow can only take questions whose shape is known in advance. 15 of the 24 qualify
(`q.shape == "fixed"`: a value or a ranking). The other 9 are open.

✏️ **Predict 1, before you run the next cell.** On the 15 fixed-shape questions: which is
more accurate, which makes fewer model calls and uses fewer tokens, and roughly how many
times fewer? Write it in the cell below.

In [8]:
PREDICTION_1 = "I think for those 15 fixed-shape questions workflow will be more accurate. Because there is  no loops and no choice of tools. And these exact 15 questions does not need any loops, they are executed with only one tool. And also workflow makes fewer model calls and uses fewer tokens. In the code above in the run_workflow model is called only once, run_agent calls model in the loop several times. For the workflow it sends short workflow prompt with questions and the answer is restricted with 80 tokens. And I think it is 3-4 times fewer tokens. Because agent's prompt is longer and it is sending it at least two times."  # ✏️ write inside the quotes, then run this cell

In [9]:
FIXED = [q for q in QUESTIONS if q.shape == "fixed"]
OPEN = [q for q in QUESTIONS if q.shape == "open"]
RUNS = {}
RUNS["bare"] = [run_agent(llm, toolbox, q, BARE) for q in QUESTIONS]
WORKFLOW = [run_workflow(llm, toolbox, q) for q in FIXED]

agent_fixed = [r for r, q in zip(RUNS["bare"], QUESTIONS) if q.shape == "fixed"]
agent_open = [r for r, q in zip(RUNS["bare"], QUESTIONS) if q.shape == "open"]
print(f"the {len(FIXED)} fixed-shape questions:")
print(summary_line("workflow", WORKFLOW))
print(summary_line("agent (bare)", agent_fixed))
print(f"\nthe {len(OPEN)} open questions (a workflow cannot take them):")
print(summary_line("agent (bare)", agent_open))
print("\nfixed-shape questions the workflow got wrong or could not answer:")
for r in WORKFLOW:
    if r.outcome != "correct":
        print(format_trace(r))

the 15 fixed-shape questions:
workflow       correct 13/15  wrong  0  no answer  2  95% [0.62, 0.96]  model calls  15  tokens   3852  seconds   15.8
agent (bare)   correct  9/15  wrong  2  no answer  4  95% [0.36, 0.80]  model calls  26  tokens  19835  seconds   70.6

the 9 open questions (a workflow cannot take them):
agent (bare)   correct  7/9  wrong  2  no answer  0  95% [0.45, 0.94]  model calls  25  tokens  19701  seconds   48.1

fixed-shape questions the workflow got wrong or could not answer:
q21: answer None -> none (first thing that went wrong: bad_args)
  model: {"kind":"lookup","region":"Nur-Sultan","indicator":"unemployment_rate","year":2022}
  tool   workflow(null) -> ERROR: Error: Unknown region 'Nur-Sultan'. Call list_regions to see the valid names.
q22: answer None -> none (first thing that went wrong: bad_args)
  model: {"kind":"lookup","region":"Chimkent","indicator":"avg_monthly_wage","year":2023}
  tool   workflow(null) -> ERROR: Error: Unknown region 'Chimkent'. C

✏️ **Write down what you see.** Which was cheaper, and by how much? Which questions did
one get wrong that the other got right, and what does the trace say went wrong? Is there
any question the workflow cannot even attempt?

In [10]:
OBSERVATION_1 = """Workflow called models fewer for almost 2 times, used tokens 5 times fewer and also it was 3 times faster.
So I was right about my predictions, but I was expecting that it will spend 3-4 times fewer tokens. But I was right in direction.
I think it is because agents prompt was way longer because it included tool descriptions and tool results.
Workflow got right 13 of 15 and agent 9 of 15. Workflow failed in q21 and q22 because of the region names.
It returned Unknown region with bad_arg error on the Nur-Sultan and Chimkent. But the data has those regions but with different names.
I thought it will fail on the Karagandy and the city of Almaty too, because in the data it is written as Karaganda and Almaty City.
But it got them right. I guess it's because they look similar. The results didn't show what questions agent failed but I am guessing it
contains 4 alias questions I don't know about the 2 other.
Workflow couldn't even attempt the 9 open questions with type growth, pp and nodata.
Because workflow cannot do several steps like getting two values and then calculating.
On these questions the agent got 7 of 9 right, the reason is it can do several steps and get the needed result."""  # ✏️ write inside the quotes, then run this cell

## Part 2 -- write the agent loop

Everything the agent does is a loop around a few calls you already have:

- `build_messages(question)` gives the starting conversation (system prompt + question);
- `llm.generate(messages)` returns a reply with `.text`;
- `parse_tool_calls(text)` returns the list of tool calls found in a reply (empty if the
  model wrote a normal answer);
- `toolbox.run(name, arguments)` runs a tool and returns its output as text, or raises
  `ToolError`.

Fill in the three TODOs. The loop is about eight lines. Then `check_loop` runs your loop
and the reference loop on three questions: with greedy decoding a correct loop gives the
same final reply and the same number of tool results.

In [11]:
def my_agent(question, max_steps=8):
    """Run one question to the end. Return (the final reply text, the list of messages)."""
    messages = build_messages(question)
    for _ in range(max_steps):
        # TODO 1 -- ask the model:  gen = llm.generate(messages)
        #           then append its text to `messages` as {"role": "assistant", "content": ...}
        gen = llm.generate(messages)
        messages.append({"role": "assistant", "content": gen.text})

        # TODO 2 -- find the tool calls in the reply:  calls = parse_tool_calls(gen.text)
        #           if there are none, the model has finished: return gen.text, messages
        calls = parse_tool_calls(gen.text)
        if not calls:
            return gen.text, messages

        # TODO 3 -- otherwise run every call with toolbox.run(call.name, call.arguments)
        #           and append each result to `messages` as {"role": "tool", "content": result}
        for call in calls:
            result = toolbox.run(call.name, call.arguments)
            messages.append({"role": "tool", "content": result})
    return "", messages

In [12]:
check_loop(my_agent, llm, toolbox, QUESTIONS)

q01: MATCHES the reference loop (tool results: yours 1, reference 1)
q07: MATCHES the reference loop (tool results: yours 1, reference 1)
q12: MATCHES the reference loop (tool results: yours 2, reference 2)
all three match


True

Your loop is the `BARE` loop, with no protection. Question `q21` asks about
"Nur-Sultan", a name the tools do not know. Run it through **your** loop.

In [13]:
q21 = next(q for q in QUESTIONS if q.qid == "q21")
print(q21.text)
try:
    my_agent(q21.text)
except ToolError as exc:
    print("your loop crashed:", exc)

What was the unemployment rate in Nur-Sultan in 2022?
your loop crashed: Unknown region 'Nur-Sultan'. Call list_regions to see the valid names.


✏️ **Write down:** what ends your loop, and what happened to `q21`? What should a loop do
with an error from a tool instead of crashing, and what could go wrong if it does that?

In [14]:
LOOP_NOTES = """Loop ends when the max_steps runs out or when the model replies without a tool call.
For q21 it ended with a ToolError. So model read the question and called get_series with region Nur-Sultan, with its indicator and the year.
My loop passed it to the toolbox.run, this tool has function _region and it searched for Nur-Sultan in the region names that is has,
and couldn't find it, so it threw ToolError.  Instead of crashing it should pass the error message to the model, it contains all information
like why it crashed and how to handle it, so model can decide what to do next.
But it might repeat the wrong call over and over until there is no steps are left, so it just wasted tokens for nothing.
Or it might pick a wrong region name."""  # ✏️ write inside the quotes, then run this cell

## Part 3 -- the harness, one layer at a time

A harness is everything around the model that is ordinary code. Three layers, each a
flag in `Harness`; every layer keeps the ones before it:

- **Layer 1, tool errors go back to the model.** A wrong region name, a bad argument or a
  broken call is reported to the model as text and it may try again. Without this layer
  the program stops at the first error.
- **Layer 2, a step budget.** 5 steps instead of 8. A repeated identical call, or the last
  step, ends tool use: the model must write its final reply from what it already has.
- **Layer 3, grounding.** The reply must end with `ANSWER: <value>`, and every number in it
  must appear in some tool output (years are exempt). If not, the harness sends the model
  back (twice at most) and finally reports `UNVERIFIED` instead of an answer.

Results have three kinds: **correct**, **wrong**, and **no answer** (the program stopped,
the budget ran out, or `UNVERIFIED`). Answers are scored by content: a reply in a sentence
counts if it contains the right value.

In [15]:
for h in (BARE, LAYER_1, LAYER_2, LAYER_3):
    print(h)

Harness(validate=False, limit=False, ground=False, max_steps=8, max_fixes=2)
Harness(validate=True, limit=False, ground=False, max_steps=8, max_fixes=2)
Harness(validate=True, limit=True, ground=False, max_steps=5, max_fixes=2)
Harness(validate=True, limit=True, ground=True, max_steps=5, max_fixes=2)


✏️ **Predict 3, before you run the next cell.** For each layer say what it will change in
the numbers (correct / wrong / no answer, out of 24) and on which kind of question. Say
also which layer you expect to change nothing at all.

In [16]:
PREDICTION_3 = """For Layer_1 I think it will change answers to 2 of 4 alias questions (Nur-Sultan, Chimkent) by calling list_regions after reading the error messages.
For Layer_2 I expect that it will stop 1 question from alias, because I think Layer_1 might repeat the same mistake over and over until it hits the limit of steps.
For Layer_3 I expect that it will fix the wrong answer to 2 questions of growth and pp. Because if the model gives a number that is not in any tool output like
it made up numbers in its head, it is sent back to use the calculate tool. If it still fails, the answer will be no answer.
I think every layer changes something, so no layer changes nothing at all"""  # ✏️ write inside the quotes, then run this cell

In [17]:
for name, harness in (("layer 1", LAYER_1), ("layer 2", LAYER_2), ("layer 3", LAYER_3)):
    RUNS[name] = [run_agent(llm, toolbox, q, harness) for q in QUESTIONS]
for name, results in RUNS.items():
    print(summary_line(name, results))
print("\nthe first thing that went wrong in each run, counted:")
for name, results in RUNS.items():
    print(f"{name:14s}", dict(Counter(r.failure for r in results if r.failure)))

bare           correct 16/24  wrong  4  no answer  4  95% [0.47, 0.82]  model calls  51  tokens  39536  seconds  118.6
layer 1        correct 18/24  wrong  6  no answer  0  95% [0.55, 0.88]  model calls  62  tokens  48708  seconds  116.4
layer 2        correct 18/24  wrong  6  no answer  0  95% [0.55, 0.88]  model calls  62  tokens  48708  seconds  122.2
layer 3        correct 18/24  wrong  4  no answer  2  95% [0.55, 0.88]  model calls  74  tokens  60116  seconds  130.8

the first thing that went wrong in each run, counted:
bare           {'no_answer_line': 7, 'bad_args': 4}
layer 1        {'no_answer_line': 7, 'bad_args': 4}
layer 2        {'no_answer_line': 7, 'bad_args': 4}
layer 3        {'no_answer_line': 7, 'ungrounded': 1, 'bad_args': 4}


In [18]:
print(outcome_table(RUNS, QUESTIONS))

id   kind         bare   layer 1   layer 2   layer 3
q01  lookup         ok        ok        ok        ok
q02  lookup         ok        ok        ok        ok
q03  lookup         ok        ok        ok        ok
q04  lookup         ok        ok        ok        ok
q05  lookup         ok        ok        ok        ok
q06  lookup         ok        ok        ok        ok
q07  rank           ok        ok        ok        ok
q08  rank        WRONG     WRONG     WRONG     WRONG
q09  rank           ok        ok        ok        ok
q10  rank           ok        ok        ok        ok
q11  rank        WRONG     WRONG     WRONG     WRONG
q12  growth         ok        ok        ok        ok
q13  growth         ok        ok        ok        ok
q14  growth         ok        ok        ok        ok
q15  growth         ok        ok        ok        ok
q16  pp             ok        ok        ok        ok
q17  pp          WRONG     WRONG     WRONG     WRONG
q18  pp          WRONG     WRONG     WRONG    

Open one run and read it. Change `TRACE_QID` and `TRACE_CONFIG` to look at others; start
with a question where two configurations disagree, and with an answer that is **wrong**
under layer 3.

In [19]:
TRACE_QID, TRACE_CONFIG = "q08", "layer 3"
by_id = {name: {r.qid: r for r in results} for name, results in RUNS.items()}
print(format_trace(by_id[TRACE_CONFIG][TRACE_QID]))
print("\nwrong even with every layer on:", [r.qid for r in RUNS["layer 3"] if r.outcome == "wrong"])

q08: answer 'Jambyl' -> wrong (first thing that went wrong: no_answer_line)
  model: <tool_call> {"name": "compare_regions", "arguments": {"indicator": "avg_monthly_wage", "year": 2024}} </tool_call>
  tool   compare_regions({"indicator": "avg_monthly_wage", "year": 2024}) -> ok: [{"region":"Astana","value":632035.0,"unit":"KZT"},{"region":"Almaty Region","value":611385.0,"unit":"KZT"},{"region":"Jetisu","value":598789.0,"unit"
  model: Jambyl
  harness: Your reply has no line 'ANSWER: <value>'. Finish with that line.
  model: ANSWER: Jambyl

wrong even with every layer on: ['q08', 'q11', 'q17', 'q21']


✏️ **Write down what the layers did.** Which layer changed the number of *correct*
answers? Which one only turned *wrong* answers into *no answer*? Which never fired? Pick
one answer that is wrong even with all three layers on: why could the harness not see
it? ("Grounded" means the number came from a tool. Does that make it right?)

In [20]:
OBSERVATION_3 = """Layer_1 was the only layer that changed the number of correct answers. After reading
the error message, the model called list_regions and fixed q23 from Karagandy to Karaganda and q24
from the city of Almaty to Almaty City. But q21 and q22 went from "no answer" to wrong. Model got
a second chance, but picked the wrong region.
Layer_3 only turned wrong answers into no answer q18 and q22 turned into UNVERIFIED. It did not fix any
answer, so correct stayed at 18. For example, in q18 the model computed (16.1 - 10.6)/100 = 0.055,
divided by 100 for no reason, and then wrote 0.55, which is not in any tool output. Layer_3 sent it
back, but the model repeated the same call and wrote 0.55 again, so the answer became UNVERIFIED.
Layer_2 never fired. Layer 1 and layer 2 columns are exactly the same. Model never reached
5 steps or repeated the same call.
Q21 was wrong even with all three layers on. Model called get_series with Nur-Sultan, got the
error, called list_regions, and chose Almaty City instead of Astana, because it did not know
that Nur-Sultan is the old name of Astana. Then it got 3.9 for Almaty City and answered 3.9%.
The harness could not see the mistake, because 3.9 really is in a tool output, so the answer is
grounded. But grounded only means the number came from a tool, doesn't mean that it is the right number.
It came from the wrong region. Layer_3 checks where a number comes from, not whether the model
chose the right region, year or calculation.
I expected Layer_1 to fix 2 of 4 alias questions about Nut-Sultan and Chimkent, but it fixed only 2 and it was not them.
I expected Layer_3 to fix wrong answers on growth and pp, but it only turned them into no answer.
Layer_2 changed nothing, which I first thought but then changed my mind."""  # ✏️ write inside the quotes, then run this cell

## Part 4 -- memory: the firm's notes

Dala Insight keeps a page of notes that every analyst reads before starting. For an agent
the simplest version is the same thing: a few lines of text added to the system prompt
for every question. Write the notes **you** would give this agent after reading its
mistakes in Part 3.

One rule: notes hold *conventions and procedures* (how names are written, which tool to
use for what), **never data values or answers**. The cell checks this and refuses a note
that contains the answer to a question.

This is not retrieval (Lab 03 fetched different text for each question). Notes ride along
with every question, whether it needs them or not, and they cost tokens every time.

In [21]:
NOTES = """Region names: use only the names from list_regions. Old names and other spellings: Nur-Sultan = Astana, Chimkent = Shymkent,
Karagandy = Karaganda, the city of Almaty = Almaty City.
compare_regions lists regions from highest to lowest. For highest take the first region, for lowest take the last region.
Percentage points: subtract the old value from the new value with calculate. Do not divide.
Percent change: calculate (new - old) / old * 100.
Always end with one line: ANSWER: <value>.
"""  # ✏️ a few lines of firm knowledge, inside the quotes

In [22]:
leaks = notes_leak(NOTES, QUESTIONS)
if not NOTES.strip():
    print("the notes are empty: write something above and run this cell again")
elif leaks:
    print("your notes contain the answer to", leaks, "- notes may hold conventions, not answers.")
else:
    RUNS["layer 3 + notes"] = [run_agent(llm, toolbox, q, LAYER_3, notes=NOTES) for q in QUESTIONS]
    print(summary_line("layer 3", RUNS["layer 3"]))
    print(summary_line("+ your notes", RUNS["layer 3 + notes"]))
    print("\nquestions whose outcome changed:")
    for before, after in zip(RUNS["layer 3"], RUNS["layer 3 + notes"]):
        if before.outcome != after.outcome:
            print(f"  {before.qid}: {before.outcome} -> {after.outcome}")

layer 3        correct 18/24  wrong  4  no answer  2  95% [0.55, 0.88]  model calls  74  tokens  60116  seconds  130.8
+ your notes   correct 22/24  wrong  1  no answer  1  95% [0.74, 0.98]  model calls  62  tokens  58343  seconds  104.8

questions whose outcome changed:
  q11: wrong -> correct
  q17: wrong -> correct
  q21: wrong -> correct
  q22: none -> correct


✏️ **Write down:** what did your notes fix, and what could they not fix? Did any question
get *worse*? What does it cost to carry these notes into every question?

In [23]:
OBSERVATION_4 = """My notes fixed 4 questions, correct went from 18 to 22 out of 24.
Q21 (Nur-Sultan) and q22 (Chimkent) became correct because the notes tell the model the old names of Astana and Shymkent.
Q11 became correct because the notes say that compare_regions lists regions from highest to lowest,
so for lowest the model must take the last one. q17 became correct because the notes say that percentage points are just the difference, without dividing.
The notes could not fix q08 and q18. q08 is still wrong even though the rule about the last region is in the notes,
and q18 is still no answer even though the notes say do not divide. So a small model does not always follow the notes.
No question got worse, all 4 changes were improvements.
The notes are added to the system prompt of every model call, even for questions that do not need them, so every prompt is longer.
But in total the run became cheaper: 62 model calls instead of 74, 58343 tokens instead of 60116, and 105 seconds instead of 131.
The model made fewer mistakes, so it didn't needed extra steps.
A weak point is that the notes only cover the 4 old names from the test. A question with another old name would not be fixed.
Also, the 95% intervals overlap ([0.55, 0.88] and [0.74, 0.98]), so with only 24 questions we cannot be sure the improvement is real."""  # ✏️ write inside the quotes, then run this cell

## What you hand in

1. **This notebook**, run from top to bottom, with every ✏️ cell filled in. The next cell
   lists any that are still empty. Download it (*File -> Download -> .ipynb*) and put it
   in your fork of the repository, replacing the template.
2. **`CONCLUSION.md`** in the same fork: three short answers, **typed by you without any AI
   tool** (no chatbot, no assistant, no autocomplete, no rewriting tool, not even for
   grammar). The instructor reads it personally. It is worth 6 of the 15 points, and every
   number in it must be one your own notebook printed.
3. Commit and push to **your fork**, then submit the URL of your last commit in Canvas.
   Late commits are flagged, not hidden.

In [24]:
for name in ('PREDICTION_0', 'PREDICTION_1', 'OBSERVATION_1', 'LOOP_NOTES', 'PREDICTION_3', 'OBSERVATION_3', 'NOTES', 'OBSERVATION_4'):
    print(f"{name:14s}", "filled in" if globals()[name].strip() else "EMPTY")

PREDICTION_0   filled in
PREDICTION_1   filled in
OBSERVATION_1  filled in
LOOP_NOTES     filled in
PREDICTION_3   filled in
OBSERVATION_3  filled in
NOTES          filled in
OBSERVATION_4  filled in
